In [1]:
# ============================================================
# Blocco 0.1 - Librerie di lavoro
# ============================================================
# Path serve per costruire percorsi di file e cartelle in modo robusto.
# E' preferibile a scrivere percorsi come semplici stringhe, perche' rende
# piu' chiaro dove vengono salvati grafici e tabelle.
from pathlib import Path

# NumPy e' la libreria di base per vettori, numeri casuali e calcoli numerici.
import numpy as np

# pandas permette di organizzare i dati in tabelle, chiamate DataFrame.
# Nel notebook useremo DataFrame per scenari simulati, frequenze e statistiche.
import pandas as pd

# matplotlib.pyplot e' la libreria usata per costruire i grafici.
import matplotlib.pyplot as plt

# ============================================================
# Blocco 0.2 - Impostazioni di visualizzazione
# ============================================================
# Queste opzioni non modificano i dati: regolano solo come pandas mostra
# le tabelle dentro Jupyter.
pd.set_option("display.precision", 6)
pd.set_option("display.max_columns", 20)

# ============================================================
# Blocco 0.3 - Cartelle di output
# ============================================================
# Path.cwd() restituisce la cartella da cui il notebook viene eseguito.
# In questo progetto il notebook si trova in 04_Codice/.
NOTEBOOK_DIR = Path.cwd()

# I grafici del progetto devono essere salvati nella cartella comune graphics/.
GRAPHICS_DIR = NOTEBOOK_DIR.parent / "graphics"

# Le tabelle numeriche della Lezione 4 vengono salvate in una sottocartella
# dedicata, cosi' restano separate dagli altri materiali del corso.
OUTPUT_DIR = NOTEBOOK_DIR / "output_Lez04"

# mkdir(..., exist_ok=True) crea le cartelle se non esistono gia'.
# Se esistono, non genera errore.
GRAPHICS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Blocco 0.4 - Parametri generali della simulazione
# ============================================================
# Il seed rende riproducibili i numeri casuali: rieseguendo il notebook
# si ottengono gli stessi scenari simulati.
seed = 202604
rng = np.random.default_rng(seed)

# M e' il numero di repliche Monte Carlo, cioe' il numero di scenari simulati.
# Versione docente: usare M elevato.
# Per una preview d'aula si puo' impostare M = 50.
M = 50_000

# V0 e' il valore iniziale del portafoglio. La perdita sara' definita come
# L = -V0 * R, dove R e' il rendimento simulato.
V0 = 100.0

# ell_base e soglie_base sono soglie di perdita usate per stimare probabilita'
# del tipo Prob(L > ell).
ell_base = 5.0
soglie_base = np.array([2.0, 5.0, 8.0, 10.0])

# ============================================================
# Blocco 0.5 - Controlli minimi sui parametri
# ============================================================
# assert verifica che una condizione sia vera. Se non lo e', Python interrompe
# l'esecuzione e mostra il messaggio indicato. Qui serve a intercettare subito
# parametri incoerenti.
assert M > 0, "Il numero di simulazioni deve essere positivo."
assert V0 > 0, "Il valore iniziale del portafoglio deve essere positivo."
assert ell_base > 0, "La soglia di perdita deve essere positiva."

# ============================================================
# Blocco 0.6 - Tabella riassuntiva dei parametri
# ============================================================
# Un DataFrame e' una tabella con righe e colonne. Qui raccogliamo i parametri
# principali in modo che il notebook documenti esplicitamente l'esperimento.
tabella_generale = pd.DataFrame({
    "parametro": ["seed", "M", "V0", "soglia_base"],
    "valore": [seed, M, V0, ell_base]
})

# In Jupyter, lasciare il nome di un oggetto nell'ultima riga della cella
# mostra automaticamente il suo contenuto.
# ------------------------------------------------------------
# Dichiarazioni per Pylance / VS Code
# ------------------------------------------------------------
# Le variabili seguenti sono create progressivamente nelle tappe del notebook.
# Le annotazioni servono solo a evitare falsi positivi reportUndefinedVariable
# nelle celle sbiancate. Non assegnano valori e non modificano l'esecuzione.

stati: np.ndarray
etichette: dict[str, str]
p: np.ndarray
mu: np.ndarray
sigma: np.ndarray
param_map: pd.DataFrame
Z: np.ndarray
freq_emp: pd.DataFrame
quantile_levels: list[float]
soglia_grid: np.ndarray
valori_M: list[int]

df: pd.DataFrame
df_parametri: pd.DataFrame
df_frequenze: pd.DataFrame
df_stat_R_stato: pd.DataFrame
controllo_segno: pd.DataFrame
statistiche_L: pd.DataFrame
df_quantili: pd.DataFrame
df_soglie: pd.DataFrame
df_soglie_grid: pd.DataFrame
df_media: pd.DataFrame
df_cond: pd.DataFrame
df_valori_cond: pd.DataFrame
df_torre: pd.DataFrame
df_quantili_stato: pd.DataFrame
df_sens_M: pd.DataFrame
df_partizioni: pd.DataFrame
df_coerenza_partizioni: pd.DataFrame
df_controllo_file: pd.DataFrame

L_sorted: np.ndarray
ecdf: np.ndarray

E_L_hat: float
E_L_theory: float
diff_MC: float
E_E_L_cond_G_hat: float
tabella_generale

,parametro,valore
0,seed,202604.0
1,M,50000.0
2,V0,100.0
3,soglia_base,5.0


In [5]:
import numpy as np
import pandas as pd

# 1. Definizione degli array NumPy
stati = np.array(["N", "V", "S"])
probabilità = np.array([0.70, 0.20, 0.10])
medie = np.array([0.004, -0.006, -0.030])
volatilità = np.array([0.020, 0.045, 0.080])

# 2. Dizionario di etichette descrittive
etichette_stati = {
    "N": "Normale (Crescita stabile)",
    "V": "Volatile (Correzione di mercato)",
    "S": "Shock (Crisi/Crollo)",
}

# 3. Creazione del DataFrame df_parametri
df_parametri = pd.DataFrame(
    {
        "Stato": stati,
        "Descrizione": [etichette_stati[s] for s in stati],
        "Probabilita": probabilità,
        "Media_Rendimento": medie,
        "Volatilitita": volatilità,
    }
).set_index("Stato")

# 4. Costruzione di param_map con indice stato (conversione in dizionario annidato)
param_map = df_parametri.to_dict(orient="index")

# 5. Controlli di integrità tramite istruzioni assert
# Controllo che la somma delle probabilità sia pari a 1 (con tolleranza per float)
assert np.isclose(
    np.sum(probabilità), 1.0
), f"La somma delle probabilità è {np.sum(probabilità)}, deve essere 1.0"

# Controllo che tutte le probabilità siano strettamente positive
assert np.all(probabilità > 0), "Tutte le probabilità devono essere maggiori di 0"

# Controllo che tutte le volatilità siano strettamente positive
assert np.all(volatilità > 0), "Tutte le volatilità devono essere maggiori di 0"

df_parametri


,Descrizione,Probabilita,Media_Rendimento,Volatilitita
Stato,,,,
N,Normale (Crescita stabile),0.7,0.004,0.020
V,Volatile (Correzione di mercato),0.2,-0.006,0.045
S,Shock (Crisi/Crollo),0.1,-0.030,0.080
